# v22b Fine-tune — Balanced F1 Backbone

Goal: train a new Qwen3-8B LoRA backbone on clean v5 data with **all-label F1 pressure**, not only macro-F1.

Decision:
- NEFTune OFF
- No oversample 2×
- Unknown 1×
- Yes 1.5×
- A/B/C/D 1×
- LoRA r=16, alpha=16, dropout=0
- epochs=2
- LR=1e-4 cosine
- enable_thinking=False

This notebook prints:
- data distribution before/after sampling
- quick sanity
- final adapter/zip check

In [ ]:
# ==================================================================
# STAGE 0 -- Install / Imports
# PATCH: single-GPU QLoRA + clean Unsloth cache before imports
# ==================================================================
import os, sys, subprocess, json, re, random, shutil, zipfile, time, math
from pathlib import Path
from collections import Counter, defaultdict

# QLoRA 4-bit training must stay on the same GPU. Using 1 GPU avoids
# Accelerate device_map mismatch on Kaggle T4x2.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["USE_TF"] = "0"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# IMPORTANT after any OOM / failed Unsloth run: clear compiled cache.
# Run this notebook from a freshly restarted kernel/session.
shutil.rmtree("/kaggle/working/unsloth_compiled_cache", ignore_errors=True)
shutil.rmtree("/root/.cache/torch_extensions", ignore_errors=True)

print("CUDA_VISIBLE_DEVICES =", os.environ.get("CUDA_VISIBLE_DEVICES"))
print("Cleaned Unsloth compiled cache")

def _pip(*pkgs):
    print("pip install", " ".join(pkgs), flush=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--break-system-packages", *pkgs], check=False)

try:
    import unsloth
except Exception:
    _pip("unsloth", "peft", "trl", "bitsandbytes", "accelerate", "safetensors", "scikit-learn")

import torch
# Keep torch/accelerate on the visible GPU 0.
if torch.cuda.is_available():
    torch.cuda.set_device(0)

from unsloth import FastLanguageModel
from datasets import Dataset
from trl import SFTTrainer
from transformers import TrainingArguments
from sklearn.model_selection import train_test_split

print("="*80)
print("IMPORT SUMMARY")
print("="*80)
print("Python:", sys.version.split()[0])
print("Torch :", torch.__version__, "CUDA=", torch.cuda.is_available(), "N_GPUS=", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
print("All imports OK")


In [ ]:
# ==================================================================
# STAGE 1 -- Config / Paths
# ==================================================================
def resolve_path(candidates, label):
    import glob, os
    hits = []
    for p in candidates:
        if any(ch in p for ch in "*?["):
            hits.extend(sorted(glob.glob(p, recursive=True)))
        else:
            hits.append(p)
    for p in hits:
        if os.path.exists(p):
            print(f"resolved {label}: {p}")
            return p
    print(f"WARNING: cannot resolve {label}; using first: {candidates[0]}")
    return candidates[0]

VERSION = "v22b"
SEED = 42

TRAIN_PATH = resolve_path([
    "/kaggle/input/datasets/nguyenminhtric/test-pipeline/Logic_Based_Educational_Queries_v5_repair_tier1_dedup_normfol_drop_logicfallacy.json",
    "/kaggle/input/test-pipeline/Logic_Based_Educational_Queries_v5_repair_tier1_dedup_normfol_drop_logicfallacy.json",
    "/kaggle/input/**/Logic_Based_Educational_Queries_v5_repair_tier1_dedup_normfol_drop_logicfallacy.json",
], "TRAIN v5")

QWEN_MODEL_ID = resolve_path([
    "/kaggle/input/models/qwen-lm/qwen-3/transformers/8b/1",
    "/kaggle/input/**/qwen-3/transformers/8b/1",
    "/kaggle/input/**/Qwen3-8B*",
], "Qwen3-8B")

OUTPUT_DIR = Path("/kaggle/working/qwen3_cot_lora_v22b_v5")
CKPT_DIR   = Path("/kaggle/working/cot_lora_ckpt_v22b_v5")
ZIP_PATH   = Path("/kaggle/working/qwen3_cot_lora_v22b_v5.zip")

# Model/train config
MAX_SEQ_LENGTH = 4096
LOAD_IN_4BIT   = True
LORA_R         = 16
LORA_ALPHA     = 16
LORA_DROPOUT   = 0.0
TARGET_MODULES = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]

EPOCHS = 2
# Safe for single T4 QLoRA. If still OOM, set BATCH_SIZE=1 and GRAD_ACCUM=8.
BATCH_SIZE = 2
GRAD_ACCUM = 4
LR = 1e-4
WARMUP_STEPS = 10
LR_SCHEDULER = "cosine"
OPTIM = "adamw_8bit"

# v22b sampling policy
OVERSAMPLE = {
    "NO": 2.0,
    "UNKNOWN": 1.0,
    "YES": 1.5,
    "A": 1.0,
    "B": 1.0,
    "C": 1.0,
    "D": 1.0,
}

V14_COT_SYSTEM = (
    "You are a careful logic tutor. Given a list of premises and a question, "
    "reason step-by-step by referencing specific premises (e.g. 'From premise 3...'). "
    "Then state your conclusion on a final line in the exact format: 'Final Answer: X' "
    "where X is one of: Yes, No, Unknown, A, B, C, or D."
)

print("CONFIG OK")
print("TRAIN_PATH:", TRAIN_PATH)
print("QWEN_MODEL_ID:", QWEN_MODEL_ID)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("OVERSAMPLE:", OVERSAMPLE)

In [ ]:
# ==================================================================
# STAGE 2 -- Load data and flatten questions
# ==================================================================
random.seed(SEED)

def norm_ans(x):
    u = str(x).strip().upper()
    if u in ("YES", "Y"): return "Yes"
    if u in ("NO", "N"): return "No"
    if u in ("UNKNOWN", "UNCERTAIN", "UNK"): return "Unknown"
    if u in ("A","B","C","D"): return u
    return str(x).strip()

def get_list_field(obj, names, default=None):
    for n in names:
        if n in obj:
            return obj[n]
    return default

with open(TRAIN_PATH, "r", encoding="utf-8") as f:
    samples = json.load(f)

flat = []
for si, s in enumerate(samples):
    premises = get_list_field(s, ["premises-NL", "premises_NL", "premises"], [])
    qs = get_list_field(s, ["questions", "question"], [])
    ans = get_list_field(s, ["answers", "answer"], [])
    exps = get_list_field(s, ["explanation", "explanations"], [])
    idxs = get_list_field(s, ["idx", "indices"], None)

    if isinstance(qs, str): qs = [qs]
    if isinstance(ans, str): ans = [ans] * len(qs)
    if isinstance(exps, str): exps = [exps] * len(qs)
    if exps is None: exps = [""] * len(qs)

    for qi, q in enumerate(qs):
        a = norm_ans(ans[qi] if qi < len(ans) else "")
        exp = exps[qi] if qi < len(exps) else ""
        idx = idxs[qi] if isinstance(idxs, list) and qi < len(idxs) else None
        flat.append({
            "sample_id": si,
            "q_idx": qi,
            "premises": premises,
            "question": q,
            "answer": a,
            "explanation": exp,
            "idx": idx,
        })

print("Records:", len(samples))
print("Questions:", len(flat))
print("Original answer distribution:")
for k,v in Counter(x["answer"] for x in flat).most_common():
    print(f"  {k:8s}: {v}")

In [ ]:
# ==================================================================
# STAGE 3 -- Split by sample_id to avoid question leakage
# ==================================================================
sample_ids = sorted(set(x["sample_id"] for x in flat))
train_ids, val_ids = train_test_split(sample_ids, test_size=0.20, random_state=SEED)
train_ids, val_ids = set(train_ids), set(val_ids)

train_flat = [x for x in flat if x["sample_id"] in train_ids]
val_flat   = [x for x in flat if x["sample_id"] in val_ids]

print("Train questions:", len(train_flat), "Val questions:", len(val_flat))
print("Train dist:", Counter(x["answer"] for x in train_flat))
print("Val dist  :", Counter(x["answer"] for x in val_flat))

In [ ]:
# ==================================================================
# STAGE 4 -- v22b oversampling: No 2x, Yes 1.5x, Unknown 1x, MC 1x
# ==================================================================
def deterministic_oversample(examples, multipliers, seed=42):
    rng = random.Random(seed)
    out = []
    by_cls = defaultdict(list)
    for e in examples:
        by_cls[e["answer"].upper()].append(e)

    for cls, arr in by_cls.items():
        m = float(multipliers.get(cls, 1.0))
        whole = int(math.floor(m))
        frac = m - whole
        for _ in range(whole):
            out.extend(arr)
        if frac > 1e-9:
            k = int(round(len(arr) * frac))
            arr2 = arr[:]
            rng.shuffle(arr2)
            out.extend(arr2[:k])
    rng.shuffle(out)
    return out

train_bal = deterministic_oversample(train_flat, OVERSAMPLE, seed=SEED)

print("="*70)
print("v22b TRAIN DISTRIBUTION BEFORE/AFTER")
print("="*70)
before = Counter(x["answer"] for x in train_flat)
after  = Counter(x["answer"] for x in train_bal)

labels = ["A","B","C","D","Yes","No","Unknown"]
for lab in labels:
    print(f"{lab:8s}: {before.get(lab,0):4d} -> {after.get(lab,0):4d}")
print("Total:", len(train_flat), "->", len(train_bal))

In [ ]:
# ==================================================================
# STAGE 5 -- Build SFT texts
# ==================================================================
def format_premises(premises):
    return "\n".join([f"Premise {i+1}: {p}" for i,p in enumerate(premises)])

def build_prompt(item):
    return (
        f"Premises:\n{format_premises(item['premises'])}\n\n"
        f"Question:\n{item['question']}\n\n"
        "Reason step-by-step by citing relevant premise numbers. "
        "End with exactly one line: Final Answer: X"
    )

def build_target(item):
    exp = str(item.get("explanation") or "").strip()
    ans = norm_ans(item["answer"])
    if exp:
        return f"{exp}\nFinal Answer: {ans}"
    return f"Final Answer: {ans}"

def make_chat_text(item, tokenizer):
    messages = [
        {"role": "system", "content": V14_COT_SYSTEM},
        {"role": "user", "content": build_prompt(item)},
        {"role": "assistant", "content": build_target(item)},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)

print("Text builders OK")

In [ ]:
# ==================================================================
# STAGE 6 -- Load Qwen + LoRA
# PATCH: fixed single-device device_map for 4-bit QLoRA
# ==================================================================
DEVICE_ID = 0
if torch.cuda.is_available():
    torch.cuda.set_device(DEVICE_ID)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = QWEN_MODEL_ID,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = None,
    load_in_4bit = LOAD_IN_4BIT,
    device_map = {"": DEVICE_ID},
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = FastLanguageModel.get_peft_model(
    model,
    r = LORA_R,
    target_modules = TARGET_MODULES,
    lora_alpha = LORA_ALPHA,
    lora_dropout = LORA_DROPOUT,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = SEED,
    use_rslora = False,
    loftq_config = None,
)

print("Model + LoRA loaded")
print("Current device:", torch.cuda.current_device() if torch.cuda.is_available() else "CPU")
print("GPU:", torch.cuda.get_device_name(torch.cuda.current_device()) if torch.cuda.is_available() else "CPU")


In [ ]:
# ==================================================================
# STAGE 7 -- Tokenize dataset
# ==================================================================
train_texts = [make_chat_text(x, tokenizer) for x in train_bal]
val_texts = [make_chat_text(x, tokenizer) for x in val_flat]

train_ds = Dataset.from_dict({"text": train_texts})
val_ds = Dataset.from_dict({"text": val_texts})

print("Train examples:", len(train_ds))
print("Val examples:", len(val_ds))
print("Example text:")
print(train_texts[0][:1000])

In [ ]:
# ==================================================================
# STAGE 8 -- Train
# PATCH: stable single-T4 training args
# ==================================================================
args = TrainingArguments(
    output_dir = str(CKPT_DIR),
    per_device_train_batch_size = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM,
    warmup_steps = WARMUP_STEPS,
    num_train_epochs = EPOCHS,
    learning_rate = LR,
    fp16 = not torch.cuda.is_bf16_supported(),
    bf16 = torch.cuda.is_bf16_supported(),
    logging_steps = 5,
    optim = OPTIM,
    weight_decay = 0.01,
    lr_scheduler_type = LR_SCHEDULER,
    seed = SEED,
    save_strategy = "epoch",
    report_to = "none",
    remove_unused_columns = False,
    gradient_checkpointing = True,
    gradient_checkpointing_kwargs = {"use_reentrant": False},
)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_ds,
    eval_dataset = None,
    dataset_text_field = "text",
    max_seq_length = MAX_SEQ_LENGTH,
    dataset_num_proc = 1,
    packing = False,
    args = args,
)

print("="*70)
print("TRAIN CONFIG CHECK")
print("="*70)
print("BATCH_SIZE:", BATCH_SIZE)
print("GRAD_ACCUM:", GRAD_ACCUM)
print("effective_batch:", BATCH_SIZE * GRAD_ACCUM)
print("MAX_SEQ_LENGTH:", MAX_SEQ_LENGTH)
print("CUDA current device:", torch.cuda.current_device() if torch.cuda.is_available() else "CPU")
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(torch.cuda.current_device()))
    print("allocated_GB_before:", torch.cuda.memory_allocated()/1024**3)
    print("reserved_GB_before :", torch.cuda.memory_reserved()/1024**3)

t0 = time.time()
train_result = trainer.train()
print("="*70)
print("TRAIN DONE")
print("="*70)
print("duration_min:", (time.time()-t0)/60)
print(train_result)


In [ ]:
# ==================================================================
# STAGE 9 -- Quick sanity on 12 val questions, greedy
# ==================================================================
FastLanguageModel.for_inference(model)

ANSWER_RE = re.compile(r"Final\s*Answer\s*:\s*(Yes|No|Unknown|A|B|C|D)", re.I)

def extract_answer(text):
    m = ANSWER_RE.search(text or "")
    if m:
        return norm_ans(m.group(1))
    # fallback: last standalone answer
    for lab in ["Unknown", "Yes", "No", "A", "B", "C", "D"]:
        if re.search(rf"\b{lab}\b", text or "", re.I):
            return norm_ans(lab)
    return "UNPARSEABLE"

def gen_one(item, max_new_tokens=256):
    messages = [
        {"role": "system", "content": V14_COT_SYSTEM},
        {"role": "user", "content": build_prompt(item)},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LENGTH).to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    text = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return text, extract_answer(text)

print("="*70)
print("QUICK SANITY (12 val questions, greedy)")
print("="*70)
san = val_flat[:12]
ok = 0
for i,item in enumerate(san, 1):
    text, pred = gen_one(item)
    gold = item["answer"]
    good = pred == gold
    ok += int(good)
    print(f"[{i:2d}/12] gold={gold:>8s} pred={pred:>11s} {'OK' if good else 'X'}")
print("-"*70)
print(f"Sanity acc: {ok}/12 = {ok/12:.0%}")
if ok >= 8:
    print("Status: OK / proceed to full eval")
elif ok >= 6:
    print("Status: borderline / full eval still useful")
else:
    print("Status: weak / inspect before full eval")

In [ ]:
# ==================================================================
# STAGE 10 -- Save adapter + zip
# ==================================================================
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))

if ZIP_PATH.exists():
    ZIP_PATH.unlink()

shutil.make_archive(str(ZIP_PATH).replace(".zip",""), "zip", root_dir=str(OUTPUT_DIR))

print("="*70)
print("SAVE CHECK")
print("="*70)
print("Adapter dir:", OUTPUT_DIR)
print("Zip path   :", ZIP_PATH)
print("Zip exists :", ZIP_PATH.exists())
if ZIP_PATH.exists():
    print("Zip MB     :", ZIP_PATH.stat().st_size / 1024**2)

for fn in ["adapter_config.json", "adapter_model.safetensors"]:
    print(fn, "OK" if (OUTPUT_DIR/fn).exists() else "MISSING")

meta = {
    "version": VERSION,
    "train_path": TRAIN_PATH,
    "base_model": QWEN_MODEL_ID,
    "oversample": OVERSAMPLE,
    "lora": {"r": LORA_R, "alpha": LORA_ALPHA, "dropout": LORA_DROPOUT, "target_modules": TARGET_MODULES},
    "train": {"epochs": EPOCHS, "batch_size": BATCH_SIZE, "grad_accum": GRAD_ACCUM, "lr": LR, "scheduler": LR_SCHEDULER, "optim": OPTIM},
    "output_dir": str(OUTPUT_DIR),
    "zip_path": str(ZIP_PATH),
}
with open("/kaggle/working/v22b_train_meta.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

print("FINETUNE v22b COMPLETED")